<!-- NB_DOC_INTRO_V1 -->
# DL — TensorFlow / Keras 3 cheat-sheet

> 📚 **Doc thematique** : [docs/04_DL.md](docs/04_DL.md) (Deep Learning)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**TensorFlow** + **Keras 3** (API recommandee 2024-2025, multi-backend : TF, PyTorch, JAX). Ce notebook execute un classifieur MNIST complet en Keras avec : Sequential API, Functional API, custom Layer, callbacks, tf.data pipeline.

Pour PyTorch : [DL_PyTorch.ipynb](./DL_PyTorch.ipynb). Pour Lightning : [DL_PyTorch_Lightning.ipynb](./DL_PyTorch_Lightning.ipynb).

## Plan

1. Setup + dataset
2. Sequential API (le plus simple)
3. Functional API (modeles non lineaires)
4. Custom Layer / Model (subclassing)
5. Callbacks (EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard)
6. Class weights pour desequilibre
7. tf.data pipeline (perf)
8. Save/load (.keras format)
9. Pieges + References


In [ ]:
# Tester si TF/Keras dispo, sinon skip + mock minimal
try:
    import tensorflow as tf
    import keras
    TF_OK = True
    print(f"TF     : {tf.__version__}")
    print(f"Keras  : {keras.__version__}")
    print(f"GPU    : {len(tf.config.list_physical_devices('GPU'))}")
except ImportError:
    TF_OK = False
    print("TF/Keras pas dispo — code en mode demo uniquement.")

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Sequential API — le plus simple

In [ ]:
if TF_OK:
    from keras import layers, models

    model = models.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    model.summary()
else:
    print("TF skip")

## 2. Demo MNIST (1 epoch pour rapidite)

In [ ]:
if TF_OK:
    # MNIST built-in Keras
    (X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
    X_tr = X_tr.reshape(-1, 784).astype("float32") / 255.0
    X_te = X_te.reshape(-1, 784).astype("float32") / 255.0

    # Subset pour rapidite (5000 train, 1000 test)
    X_tr, y_tr = X_tr[:5000], y_tr[:5000]
    X_te, y_te = X_te[:1000], y_te[:1000]
    print(f"Train : {X_tr.shape}, Test : {X_te.shape}")

    history = model.fit(X_tr, y_tr, epochs=2, batch_size=64,
                        validation_split=0.1, verbose=0)
    test_loss, test_acc = model.evaluate(X_te, y_te, verbose=0)
    print(f"\nTest accuracy : {test_acc:.4f}")
else:
    print("TF skip")

## 3. Functional API — modeles non lineaires (multi-input, skip connections)

In [ ]:
if TF_OK:
    inputs = keras.Input(shape=(784,))
    x = layers.Dense(128, activation="relu")(inputs)
    x = layers.Dropout(0.3)(x)
    # Skip connection
    skip = layers.Dense(64, activation="relu")(x)
    y = layers.Dense(64, activation="relu")(x)
    merged = layers.Add()([skip, y])
    outputs = layers.Dense(10, activation="softmax")(merged)

    model_func = keras.Model(inputs, outputs, name="mlp_with_skip")
    model_func.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    print(f"Params : {model_func.count_params()}")
else:
    print("TF skip")

## 4. Custom Layer (subclassing)

In [ ]:
if TF_OK:
    class GatedLinear(layers.Layer):
        """Custom layer : x → linear(x) * sigmoid(gate(x))"""
        def __init__(self, units, **kwargs):
            super().__init__(**kwargs)
            self.units = units

        def build(self, input_shape):
            self.w = self.add_weight(shape=(input_shape[-1], self.units),
                                      initializer="he_normal", name="w")
            self.b = self.add_weight(shape=(self.units,), initializer="zeros", name="b")
            self.g = self.add_weight(shape=(input_shape[-1], self.units),
                                      initializer="he_normal", name="g")

        def call(self, x):
            linear = x @ self.w + self.b
            gate = keras.activations.sigmoid(x @ self.g)
            return linear * gate

    model_custom = keras.Sequential([
        keras.Input(shape=(784,)),
        GatedLinear(64),
        layers.Dense(10, activation="softmax"),
    ])
    model_custom.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
    print(f"Custom layer model params : {model_custom.count_params()}")
else:
    print("TF skip")

## 5. Callbacks — EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [ ]:
if TF_OK:
    import tempfile, os

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(tempfile.gettempdir(), "best_model.keras"),
            monitor="val_accuracy", save_best_only=True, verbose=0,
        ),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
        keras.callbacks.TensorBoard(log_dir=os.path.join(tempfile.gettempdir(), "tb_logs")),
    ]

    # Re-utiliser le model Sequential
    history2 = model.fit(X_tr, y_tr, epochs=3, batch_size=64,
                         validation_split=0.1, callbacks=callbacks, verbose=0)
    print(f"Final val_acc : {history2.history['val_accuracy'][-1]:.4f}")
else:
    print("TF skip")

## 6. Class weights pour desequilibre

In [ ]:
if TF_OK:
    from sklearn.utils.class_weight import compute_class_weight

    # Simuler desequilibre : garder 50 samples de chaque classe sauf 0 (200)
    weights = compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)
    class_weight_dict = dict(enumerate(weights))
    print(f"Class weights : {class_weight_dict}")

    # On peut maintenant passer class_weight=... a model.fit()
    # model.fit(X_tr, y_tr, class_weight=class_weight_dict, ...)
else:
    print("TF skip")

## 7. tf.data pipeline (perf : prefetch, cache, parallel map)

In [ ]:
if TF_OK:
    BATCH_SIZE = 64

    train_ds = (tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
                .shuffle(1000, seed=SEED)
                .batch(BATCH_SIZE)
                .prefetch(tf.data.AUTOTUNE))    # prefetch = overlap CPU prep + GPU compute

    test_ds = (tf.data.Dataset.from_tensor_slices((X_te, y_te))
               .batch(BATCH_SIZE)
               .prefetch(tf.data.AUTOTUNE))

    # Cache si dataset tient en RAM
    # train_ds = train_ds.cache()

    print(f"Train dataset : {len(list(train_ds))} batches")
    model.fit(train_ds, validation_data=test_ds, epochs=1, verbose=0)
    print("Fit avec tf.data OK")
else:
    print("TF skip")

## 8. Save / Load — format `.keras`

In [ ]:
if TF_OK:
    import tempfile, os
    path = os.path.join(tempfile.gettempdir(), "demo_model.keras")
    model.save(path)
    print(f"Sauve : {path}")

    loaded = keras.models.load_model(path)
    loss, acc = loaded.evaluate(X_te, y_te, verbose=0)
    print(f"Loaded accuracy : {acc:.4f}")
else:
    print("TF skip")

## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| `categorical_crossentropy` quand y entier | `sparse_categorical_crossentropy` |
| Pas de `restore_best_weights` dans EarlyStopping | Final = derniere epoch (souvent overfit) |
| Sigmoid sur output multi-class | softmax + categorical_crossentropy |
| Pas de validation_split / validation_data | Pas de detection overfitting |
| Format `.h5` (legacy) | `.keras` (recommande Keras 3) |
| Pas de tf.data prefetch | GPU sub-utilise |
| Pas de class_weight sur desequilibre | F1 effondre |
| Pas de seed | Reproductibilite cassee — `keras.utils.set_random_seed(42)` |


## References

### Documentation
- Keras docs : https://keras.io/
- TF guide : https://www.tensorflow.org/guide
- Keras 3 multi-backend : https://keras.io/keras_3/

### Voir aussi
- [DL_Deep_Learning_Maths.ipynb](DL_Deep_Learning_Maths.ipynb)
- [DL_PyTorch.ipynb](DL_PyTorch.ipynb)
- [DL_PyTorch_Lightning.ipynb](DL_PyTorch_Lightning.ipynb)
